In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import pyremo as pr
import os
from openpyxl import load_workbook
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from REMO2020TEB_mod import flip, parse_paris_time, get_paris_var_index, parisclass, modelclass_cities, prepare_remo_to_umask_seas

In [ ]:
# give figure path
figpath = './figures/'

In [ ]:
# Give the quantiles to be calculated (must inlcude 0.5)
quantiles = np.array([0.05,0.25,0.5,0.75,0.95])

In [ ]:
# if we want to replace median with mean
lreplace_median=True

In [ ]:
# Measurement data for Paris
paris_datapath = "./data_in/measurement_data/Paris/"
paris_data_nc = xr.open_dataset(paris_datapath+"Obs_MF_01_to_12_2020.nc")
paris_data_nc = parse_paris_time(paris_data_nc) # change time to datetime format
paris_data_xlsx = load_workbook(filename = paris_datapath+"Obs_MF.xlsx")

In [ ]:
# Loop over different stations (51 currently)
paris_data =[] # init
pxls = paris_data_xlsx['Liste_Instruments']
for ii in range(51):
    paris_data.append(parisclass(int(pxls['D'+str(ii+2)].value),pxls['E'+str(ii+2)].value,float(pxls['F'+str(ii+2)].value),float(pxls['G'+str(ii+2)].value),pxls['K'+str(ii+2)].value))

In [ ]:
# Stations to be used
paris_urb_stats = np.array(["Porte-de-Versaille","Longchamp-Bois-De-Boulogne"])
paris_rur_stats = np.array(["Changis","Courdimanche","Dourdan","Melun","Pontoise","Prunay-Le-Temple"])

In [ ]:
# Get all urban indexes (not Eifel-Tower) and rural indexes
paris_urb_ind = np.array([], dtype="int32")
paris_rur_ind = np.array([], dtype="int32")
for obj in paris_data:
    if(obj.vartype == "Urban" and any(paris_urb_stats[:] == obj.sitename) and obj.sitename != "Eiffel-Tower" and obj.sitename != "Paris-Montsouris-double"):
        paris_urb_ind = np.append(paris_urb_ind,obj.varind-1) # to python indexing
    if((obj.vartype == "Crops" or obj.vartype == "Forest") and any(paris_rur_stats[:] == obj.sitename)):
        paris_rur_ind = np.append(paris_rur_ind,obj.varind-1) # to python indexing

In [ ]:
# calculate urban and rural means over the stations, and make seasonal diurnal means
paris_wsp_urban_raw = get_paris_var_index(paris_data_nc,"WIND_FORCE",paris_urb_ind,make_mean=True).groupby(["season", "hour"])
paris_wsp_urban = paris_wsp_urban_raw.quantile(quantiles, dim='time')
paris_wsp_rural_raw = get_paris_var_index(paris_data_nc,"WIND_FORCE",paris_rur_ind,make_mean=True).groupby(["season", "hour"])
paris_wsp_rural = paris_wsp_rural_raw.quantile(quantiles, dim='time')
if (lreplace_median):
    paris_wsp_urban.loc[dict(quantile=0.5)] = paris_wsp_urban_raw.mean("time")
    paris_wsp_rural.loc[dict(quantile=0.5)] = paris_wsp_rural_raw.mean("time")

In [ ]:
# Measurement data for Paris
#paris_datapath2 = "/work/ch0636/g300096/data/Paris/"
#paris_data_nc2 = xr.open_dataset(paris_datapath2+"75114001_PARIS-MONTSOURIS_MTO_1H_2020.nc")
#paris_data_nc2 = paris_data_nc2.assign_coords(season=("time", paris_data_nc2.time.dt.season.data),
#    hour=("time", paris_data_nc2.time.dt.hour.data))
#paris_wsp_urban_raw = paris_data_nc2["ws"].groupby(["season", "hour"])
#paris_wsp_urban = paris_wsp_urban_raw.quantile(quantiles, dim='time')
#if (lreplace_median):
#    paris_wsp_urban.loc[dict(quantile=0.5)] = paris_wsp_urban_raw.mean("time")

In [ ]:
# read in REMO hourly data
remo_datap = "/work/ch0636/g300096/results/TEB/post-processed"

In [ ]:
# read in urban/rural mask
urb_rur_remo_mask_hr = xr.open_dataset("./data_in/model_data/REMO_static/urmask_Paris-ALPX-3_urclim_masks_fx.nc")
urb_rur_remo_mask_lr = xr.open_dataset("./data_in/model_data/REMO_static/urmask_Paris-EUR-11_urclim_masks_fx.nc")

In [ ]:
# define the runs to be analysed
modelruns =[] # init

# 
modelruns.append(modelclass_cities("036047","REMO2020 TEB","ECOSG_LCZ",'b-','0.0275'))
modelruns.append(modelclass_cities("036048","REMO2020 TEB iMOVE","LANDMATE",'r-','0.0275'))
modelruns.append(modelclass_cities("036049","REMO2020","GLCC",'c-','0.0275'))
modelruns.append(modelclass_cities("036043","REMO2020 TEB","ECOSG_LCZ",'b--','0.11'))
modelruns.append(modelclass_cities("036044","REMO2020 TEB iMOVE","LANDMATE",'r--','0.11'))
modelruns.append(modelclass_cities("036041","REMO2020","GLCC",'c--','0.11'))

In [ ]:
# read in data
for obj in modelruns:
    if (obj.reso == '0.0275'):
        obj.rawdata = xr.merge([pr.parse_dates(xr.open_dataset(remo_datap+"/"+obj.userexp+"_postpro_1h_2020.nc")),urb_rur_remo_mask_hr[obj.maskname]], compat="override", join="override")
    elif (obj.reso == '0.11'):
        obj.rawdata = xr.merge([pr.parse_dates(xr.open_dataset(remo_datap+"/"+obj.userexp+"_postpro_1h_2020.nc")),urb_rur_remo_mask_lr[obj.maskname]], compat="override", join="override")
    # Shift time by 30 minutes backwards (REMO specific)
    obj.rawdata = obj.rawdata.assign_coords(time=obj.rawdata.time - pd.Timedelta('30min'))
    obj.rawdata = obj.rawdata.assign_coords(
        season=obj.rawdata.time.dt.season,
        hour=obj.rawdata.time.dt.hour
    )

In [ ]:
# prepare wanted variable
varname="WIND10"
for obj in modelruns:
    obj.vardata = prepare_remo_to_umask_seas(obj.rawdata,obj.maskname,varname,lreplace_median,quantiles)

In [ ]:
# Define the order of seasons to be used (also when plotting)
seasons = ["DJF","MAM","JJA","SON"]

In [ ]:
# select plotted quantiles for shading (and 1-q)
ploquan = [0.05,0.25]

### Prepare some common plotting configs

In [ ]:
# Reset and set rcParams
mpl.rcParams.update(mpl.rcParamsDefault)
#plt.rcParams["figure.figsize"] = (25, 25)
plt.rcParams["figure.subplot.wspace"] = 0.3
plt.rcParams["axes.titlesize"] = 26
plt.rcParams['axes.labelsize'] = 22
plt.rcParams["xtick.top"] = True
plt.rcParams["xtick.labeltop"] = False
plt.rcParams["xtick.bottom"] = True
plt.rcParams["xtick.labelbottom"] = True
plt.rcParams["ytick.left"] = True
plt.rcParams["ytick.labelleft"] = True
plt.rcParams["ytick.right"] = True
plt.rcParams["ytick.labelright"] = False
plt.rcParams['lines.markersize'] = 10
plt.rc('text', usetex=True)
plt.rc('font', family='serif')
plt.rc('xtick', labelsize=24) 
plt.rc('ytick', labelsize=24) 
plt.rc('legend', fontsize=22)

# Prepare wind speed plots

In [ ]:
# plot specific configs
plt.rcParams["figure.subplot.hspace"] = 0.9
# Create subplots: 2 rows, 4 columns
fig, ax = plt.subplots(2, 4, figsize=(25, 10))
ax = ax.flatten()  # Flatten for easy indexing

# number of legend columns
ncol = 4
# some helper indexes
ind = 0
for seas in seasons:
    ax[ind].grid() # make grid and modify it
    ax[ind].grid(which='minor', color='lightgrey', linestyle=':', linewidth=0.5)
    ax[ind].minorticks_on() # use minor ticks
    ax[ind + len(seasons)].grid() # make grid and modify it
    ax[ind + len(seasons)].grid(which='minor', color='lightgrey', linestyle=':', linewidth=0.5)
    ax[ind + len(seasons)].minorticks_on() # use minor ticks
    # Urban plot
    ax[ind].plot(paris_wsp_urban.hour.values+0.5,
                 paris_wsp_urban.sel(season=seas).sel(quantile=0.5).values,
                 'k-', label="Measurements")
    for qq in ploquan:
        ax[ind].fill_between(paris_wsp_urban.hour.values+0.5,
                            paris_wsp_urban.sel(season=seas).sel(quantile=qq).values, paris_wsp_urban.sel(season=seas).sel(quantile=1.0-qq).values,
                            color='k', alpha=0.1, linewidth=1, linestyle = '--')
    
    for obj in modelruns:
        ax[ind].plot(obj.vardata.hour.values+0.5,
                     obj.vardata.urban.sel(season=seas).sel(quantile=0.5).values,
                     obj.plstyle, label=obj.runame+" ("+obj.reso+")")

#        for qq in ploquan:
#            ax[ind].fill_between(obj.vardata.hour.values,
#                                obj.vardata.urban.sel(season=seas).sel(quantile=qq).values, obj.vardata.urban.sel(season=seas).sel(quantile=1.0-qq).values,
#                                color=obj.plstyle[0], alpha=0.1, linewidth=1, linestyle = '--')
    ax[ind].set_title(f"Urban {seas}")
    ax[ind].set_xlabel("Hour")
    ax[ind].set_xlim([0,24])
    if seas == "DJF":
        ax[ind].set_ylabel("Wind speed [m/s]")
        # flip and insert one empty string
        handles, labels = ax[ind].get_legend_handles_labels()
        empty_patch = mpatches.Patch(color='none', label='') # create a patch with no color
        handles.insert(ncol, empty_patch)
        labels.insert(ncol,'')
        ax[ind].legend(flip(handles, ncol), flip(labels, ncol),loc=(0.05, -0.7), ncol=ncol, frameon=True)
    
    # Rural plot
    ax[ind + len(seasons)].plot(paris_wsp_rural.hour.values+0.5,
                                paris_wsp_rural.sel(season=seas).sel(quantile=0.5).values,
                                'k-', label="Measurements")
    for qq in ploquan:
        ax[ind + len(seasons)].fill_between(paris_wsp_rural.hour.values+0.5,
                            paris_wsp_rural.sel(season=seas).sel(quantile=qq).values, paris_wsp_rural.sel(season=seas).sel(quantile=1.0-qq).values,
                            color='k', alpha=0.1, linewidth=1, linestyle = '--')
    for obj in modelruns:
        ax[ind + len(seasons)].plot(obj.vardata.hour.values+0.5,
                                    obj.vardata.rural.sel(season=seas).sel(quantile=0.5).values,
                                    obj.plstyle, label=obj.runame+" ("+obj.reso+")")
#        for qq in ploquan:
#            ax[ind + len(seasons)].fill_between(obj.vardata.hour.values,
#                                obj.vardata.rural.sel(season=seas).sel(quantile=qq).values, obj.vardata.rural.sel(season=seas).sel(quantile=1.0-qq).values,
#                                color=obj.plstyle[0], alpha=0.1, linewidth=1, linestyle = '--') 
    ax[ind + len(seasons)].set_title(f"Rural {seas}")
    ax[ind + len(seasons)].set_xlabel("Hour")
    ax[ind + len(seasons)].set_xlim([0,24])
    if seas == "DJF":
        ax[ind + len(seasons)].set_ylabel("Wind speed [m/s]")
    # increase counter
    ind += 1
    
plt.savefig(figpath + 'fig7.png', format="png", bbox_inches='tight', pad_inches=0)
plt.show()